In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv("youtube_shorts_tiktok_trends_2025.csv")

df.head()

In [ ]:
print(df.shape)

df.info()

In [ ]:
df.isnull().sum()

In [ ]:
required_for_label = [
    "completion_rate",
    "engagement_velocity",
    "avg_watch_time_sec",
    "duration_sec"
]

for col in required_for_label:
    if col not in df.columns:
        raise ValueError(f"Required column missing: {col}")

In [ ]:
df["completion_rate"] = df["completion_rate"].fillna(
    df["completion_rate"].median()
)

df["engagement_velocity"] = df["engagement_velocity"].fillna(
    df["engagement_velocity"].median()
)

df["avg_watch_time_sec"] = df["avg_watch_time_sec"].fillna(
    df["avg_watch_time_sec"].median()
)

df["duration_sec"] = df["duration_sec"].replace(
    0,
    np.nan
)

df["duration_sec"] = df["duration_sec"].fillna(
    df["duration_sec"].median()
)

In [ ]:
df["watch_ratio"] = (
    df["avg_watch_time_sec"]
    /
    df["duration_sec"]
)

df["watch_ratio"] = df["watch_ratio"].replace(
    [np.inf, -np.inf],
    np.nan
)

df["watch_ratio"] = df["watch_ratio"].fillna(
    df["watch_ratio"].median()
)

In [ ]:
df["completion_rank"] = (
    df["completion_rate"]
    .rank(pct=True)
)

df["engagement_rank"] = (
    df["engagement_velocity"]
    .rank(pct=True)
)

df["watch_ratio_rank"] = (
    df["watch_ratio"]
    .rank(pct=True)
)

In [ ]:
df["viral_score"] = (
      0.45 * df["completion_rank"]
    + 0.40 * df["engagement_rank"]
    + 0.15 * df["watch_ratio_rank"]
)

In [ ]:
cutoff = df["viral_score"].quantile(0.80)

df["trend_label"] = (
    df["viral_score"] >= cutoff
).astype(int)

print(df["trend_label"].value_counts())

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(x=df["trend_label"])

plt.title("Viral vs Non-Viral Videos")
plt.xlabel("Trend Label")
plt.ylabel("Count")

plt.show()

In [ ]:
logit_features = [
    "platform",
    "category",
    "duration_sec",
    "completion_rate",
    "engagement_velocity",
    "upload_hour",
    "has_emoji"
]

logit_df = df[
    logit_features + ["trend_label"]
].copy()

In [ ]:
logit_df = logit_df.dropna()

In [ ]:
X_logit = logit_df[logit_features]

y_logit = logit_df["trend_label"]

In [ ]:
categorical_features = [
    "platform",
    "category"
]

In [ ]:
numerical_features = [
    "duration_sec",
    "completion_rate",
    "engagement_velocity",
    "upload_hour",
    "has_emoji"
]

In [ ]:
preprocessor_log = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            SimpleImputer(
                strategy="median"
            ),
            numerical_features
        )
    ]
)

In [ ]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_logit,
    y_logit,
    test_size=0.4,
    random_state=98
)

In [ ]:
log_model = Pipeline(
    steps=[
        (
            "preprocess",
            preprocessor_log
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

log_model.fit(
    X_train_log,
    y_train_log
)

In [ ]:
y_pred_log = log_model.predict(
    X_test_log
)

In [ ]:
accuracy = accuracy_score(
    y_test_log,
    y_pred_log
)

print("Accuracy:", accuracy)

In [ ]:
cm = confusion_matrix(
    y_test_log,
    y_pred_log
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(
    classification_report(
        y_test_log,
        y_pred_log
    )
)

In [ ]:
linear_features = [
    "duration_sec",
    "avg_watch_time_sec",
    "platform",
    "category",
    "event_season",
    "traffic_source"
]

linear_features = [
    col for col in linear_features
    if col in df.columns
]

linear_df = df[
    linear_features + ["completion_rate"]
].copy()

linear_df.head()

In [ ]:
linear_df = linear_df.dropna()

print(linear_df.shape)

In [ ]:
X_lin = linear_df[linear_features]

y_lin = linear_df["completion_rate"]

In [ ]:
cat_features_lin = [
    "platform",
    "category",
    "event_season",
    "traffic_source"
]

In [ ]:
num_features_lin = [
    "duration_sec",
    "avg_watch_time_sec"
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

preprocessor_lin = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_features_lin
        ),
        (
            "num",
            SimpleImputer(strategy="median"),
            num_features_lin
        )
    ]
)

In [ ]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(
    X_lin,
    y_lin,
    test_size=0.4,
    random_state=98
)

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(
    steps=[
        ("preprocess", preprocessor_lin),
        ("model", LinearRegression())
    ]
)

linear_model.fit(
    X_train_lin,
    y_train_lin
)

In [ ]:
y_pred_lin = linear_model.predict(
    X_test_lin
)

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    y_test_lin,
    y_pred_lin
)

print("MAE:", mae)

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(
    y_test_lin,
    y_pred_lin
)

print("MSE:", mse)

In [ ]:
rmse = np.sqrt(mse)

print("RMSE:", rmse)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(
    y_test_lin,
    y_pred_lin
)

print("R² Score:", r2)

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    y_test_lin,
    y_pred_lin,
    alpha=0.5
)

plt.xlabel("Actual Completion Rate")
plt.ylabel("Predicted Completion Rate")
plt.title("Actual vs Predicted Completion Rate")

plt.show()

In [ ]:
residuals = y_test_lin - y_pred_lin

plt.figure(figsize=(8,5))
plt.scatter(y_pred_lin, residuals, alpha=0.5)

plt.axhline(y=0, linestyle='--')

plt.xlabel("Predicted Completion Rate")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    residuals,
    bins=30
)

plt.xlabel("Residual Error")
plt.ylabel("Frequency")
plt.title("Distribution of Residual Errors")

plt.show()

In [ ]:
print("=== MODEL RESULTS ===")

print(f"Logistic Regression Accuracy: {accuracy:.4f}")

print(f"Linear Regression R2 Score: {r2:.4f}")

print(f"Linear Regression RMSE: {rmse:.4f}")

print("\nConclusion:")
print("The Logistic Regression model successfully predicts viral trends.")
print("The Linear Regression model accurately predicts completion rate.")
print("These models can help Nike optimize future influencer campaigns.")